# Create Mitacs Awards

Creates Mitacs awards from Mitacs' own public list of completed projects at
https://www.mitacs.ca/projects/ (WordPress; ~31,100 projects across Accelerate,
Elevate, Globalink Research Award and Business Strategy Internship). Each
project record gives the title, description, Faculty Supervisor, Student
(rarely filled), Partner organization, Discipline, Sector, University and
Program. The listing pages render every record inline, so the script reads the
~2,595 listing pages rather than 31k detail pages. **31,132 projects: 100% title, 98% description, 98% supervisor (lead), 98% university** (local run
2026-09-30).

**What Mitacs does not publish:** award / reference numbers, start or end
dates, or amounts. So:
- `amount`, `currency`, `start_date`, `end_date` are NULL by design (the §6.7
  amount check is waived: Mitacs publishes no per-project amounts).
- `funder_award_id` is the project's public URL slug
  (`https://www.mitacs.ca/our-projects/{slug}/`), the only public-facing
  per-project reference (§2.1.1 rule 2). The citation side for F4320322675
  overwhelmingly uses Mitacs internal references (`IT12345` Accelerate,
  `FR12345` Elevate; ~3,900 of ~5,500 shells), which Mitacs never prints on
  its site, so these rows will NOT collapse onto the existing shells. If Mitacs
  ever exposes the IT/FR number, switch the key to it.
- The WordPress post date is the 2026-05-27 site re-import, not an award date,
  so it is deliberately not used.

`lead_investigator` = the Faculty Supervisor (Mitacs' academic award holder of
record); `investigators` = all listed supervisors, then any listed students.

**Prerequisites:**
- Run `scripts/local/mitacs_to_s3.py` first. It uploads
  `s3://openalex-ingest/awards/mitacs/mitacs_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320322675`
- display_name: Mitacs
- ror_id / doi: from `openalex.funders.funders` (ror 00cjrc276, doi 10.13039/501100004489)

**Priority:** `481` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mitacs_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mitacs/mitacs_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.mitacs_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.mitacs_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.mitacs_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320322675 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320322675;

## Step 2: Create Mitacs Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mitacs_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320322675  -- Mitacs
),

people AS (
    -- Supervisors (in listed order) then students -> investigator structs.
    -- Names were split in Python (split_name, runbook §2.4.1).
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        funder_award_id,
        TRANSFORM(
            FILTER(
                concat(
                    COALESCE(from_json(supervisors_json, 'array<struct<name:string,given_name:string,family_name:string>>'), array()),
                    COALESCE(from_json(students_json, 'array<struct<name:string,given_name:string,family_name:string>>'), array())
                ),
                p -> NULLIF(TRIM(p.family_name), '') IS NOT NULL
            ),
            p -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', named_struct(
                    'name', NULLIF(TRIM(university), ''),
                    'country', CASE WHEN NULLIF(TRIM(university), '') IS NOT NULL THEN 'CA' END,
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                )
            )
        ) AS investigators,
        -- lead = first supervisor only (never a student)
        CASE WHEN NULLIF(TRIM(lead_family_name), '') IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(lead_given_name), ''),
            'family_name', NULLIF(TRIM(lead_family_name), ''),
            'orcid', CAST(NULL AS STRING),
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(university), ''),
                'country', CASE WHEN NULLIF(TRIM(university), '') IS NOT NULL THEN 'CA' END,
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead_investigator
    FROM openalex.awards.mitacs_raw
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CAST(NULL AS DOUBLE) as amount,
    CAST(NULL AS STRING) as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    CASE
        WHEN LOWER(g.program) LIKE '%elevate%' THEN 'fellowship'
        WHEN LOWER(g.program) LIKE '%globalink%' OR LOWER(g.program) LIKE '%business strategy%' THEN 'training'
        ELSE 'research'
    END as funding_type,
    NULLIF(TRIM(g.program), '') as funder_scheme,
    'mitacs_projects' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    CAST(NULL AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    p.lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.mitacs_raw g
JOIN people p ON p.funder_award_id = g.funder_award_id
CROSS JOIN src_funder f
WHERE g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
  -- Exclude Business Strategy Internship (BSI): business-student work placements
  -- (e.g. 'Accountant (MBSI)'), not research; listed 'supervisors' are programme
  -- coordinators. 5,142 rows as of 2026-10-01. Same non-research rule as FFG subsidies.
  AND COALESCE(TRIM(g.program), '') != 'Business Strategy Internship';

In [ ]:
%sql
-- Shape check (§2.1.1): funder_award_id is the URL slug, never a WP post id or
-- a Mitacs IT/FR internal ref (neither is shipped), and ids are unique.
SELECT
    assert_true(COUNT_IF(NOT funder_award_id RLIKE '^[a-z0-9%._-]+$' OR funder_award_id RLIKE '^[0-9]+$') = 0,
                'funder_award_id is not a URL slug') AS shape_ok,
    assert_true(COUNT(*) = COUNT(DISTINCT id), 'duplicate award ids') AS unique_ok
FROM openalex.awards.mitacs_awards;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mitacs_projects' AND priority = 481;

-- Insert into openalex_awards_raw with priority.
-- Priority 481: direct-from-funder ingest of Mitacs' own completed-projects list.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    481 as priority
FROM openalex.awards.mitacs_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_mitacs_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.mitacs_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, size(investigators) as n_people
FROM openalex.awards.mitacs_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt
FROM openalex.awards.mitacs_awards
GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail
-- (prolific Mitacs supervisors legitimately hold dozens of internships).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.mitacs_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.mitacs_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Coverage (no amounts/dates by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(funder_scheme) as has_program
FROM openalex.awards.mitacs_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mitacs_projects'
GROUP BY provenance, priority;